# BPE Token Length EDA — run this on Google Colab

## Why this notebook exists

Our local EDA (`01_data_exploration.ipynb`) measures document length by **whitespace-splitting**. That's fine for TF-IDF but it is NOT what NepBERTa sees. NepBERTa uses WordPiece / BPE subword tokenization, which typically produces 1.5× – 3× more tokens per document than whitespace splitting — and the exact ratio depends on how well NepBERTa's vocabulary covers Nepali morphology.

To set `NEPBERTA_MAX_LENGTH` honestly we need the REAL token lengths. That means running NepBERTa's tokenizer over the data. We don't want to install the full `transformers` stack (~500MB+ of PyTorch deps) locally just for tokenization, so we offload this step to Colab where it's free and fast.

## What this notebook produces

One small CSV — `bpe_length_stats.csv` — with percentile summaries (mean, median, 90th, 95th, 99th pct, max) and the fraction of rows that would be truncated at common `max_length` cutoffs (64, 128, 256). You download that file, drop it into `outputs/results/` in the repo, and the consumer cell in `01_data_exploration.ipynb` picks it up and shows the numbers.

## How to run

1. Open this notebook in Google Colab (File → Open notebook → GitHub/Upload, or just upload the .ipynb).
2. Runtime → Change runtime type → **CPU** is fine (no GPU needed, tokenization is cheap).
3. Run the cells top-to-bottom. You'll be prompted to upload the raw CSV.
4. Download the output CSVs when the last cell finishes.

## Step 1 — install the `transformers` library

Colab already has `pandas`, `numpy`, `tqdm`. We just need `transformers` for the NepBERTa tokenizer. `-q` = quiet (hide install logs).

In [ ]:
!pip install -q transformers

## Step 2 — upload the raw CSV

Running the cell below pops a file picker. Select `aayamoza.csv` from your local `data/raw/` folder. The file lands in Colab's working directory.

⚠️ Alternative: if you've already mounted Google Drive, skip this cell and edit the path in the next cell to point at your Drive location.

In [ ]:
from google.colab import files

# `files.upload()` returns a dict {filename: bytes} of what the user selected.
# We don't actually use the dict — we just need the file to land on disk.
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Step 3 — load NepBERTa's tokenizer and measure lengths

`AutoTokenizer.from_pretrained(...)` downloads NepBERTa's tokenizer (vocabulary + merge rules) from Hugging Face — first run takes ~30s, subsequent runs use the cache.

We then batch-tokenize the dataset. `tokenizer(list_of_strings, add_special_tokens=True, truncation=False)` returns a dict with `input_ids` = a list of lists of integer token IDs. We don't care about the IDs themselves, just the LENGTH of each list. `add_special_tokens=True` includes the `[CLS]` / `[SEP]` markers that fine-tuning will add — so the count reflects what the model actually processes. `truncation=False` prevents silent shortening (we want to SEE how long documents really are).

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# ~500MB download on first run (model config + tokenizer vocab).
tokenizer = AutoTokenizer.from_pretrained('NepBERTa/NepBERTa')
print('Tokenizer loaded. Vocab size:', tokenizer.vocab_size)

# Load the raw CSV (same file as the local EDA).
primary = pd.read_csv('aayamoza.csv')
print(f'Rows: {len(primary):,}')

# Coerce NaNs / non-strings to '' so the tokenizer doesn't crash.
texts = [s if isinstance(s, str) else '' for s in primary['Sentences']]

# Batch-tokenize. On CPU this takes ~5-15s for 35k short texts.
enc = tokenizer(texts, add_special_tokens=True, truncation=False)
bpe_lengths = pd.Series([len(ids) for ids in enc['input_ids']])

print('\nBPE lengths computed.')
print(bpe_lengths.describe(percentiles=[.5, .9, .95, .99]).round(1))

## Step 4 — summarise and save

We save TWO things:

- **`bpe_length_stats.csv`** — one row with percentile summaries and truncation fractions at common cutoffs. This is the headline artifact — drop it into `outputs/results/` in the repo.
- **`bpe_lengths.csv`** — per-row length file, optional. Useful if you want to re-plot a histogram locally using the real token counts. Drop into `outputs/results/` alongside the summary.

In [ ]:
# Build one summary row for the BPE length distribution.
summary = {
    'dataset':       'primary',
    'n_rows':        len(bpe_lengths),
    'mean':          round(float(bpe_lengths.mean()), 2),
    'median':        float(bpe_lengths.median()),
    'p90':           float(bpe_lengths.quantile(0.90)),
    'p95':           float(bpe_lengths.quantile(0.95)),
    'p99':           float(bpe_lengths.quantile(0.99)),
    'max':           int(bpe_lengths.max()),
    'pct_over_64':   round(float((bpe_lengths > 64).mean()  * 100), 2),
    'pct_over_128':  round(float((bpe_lengths > 128).mean() * 100), 2),
    'pct_over_256':  round(float((bpe_lengths > 256).mean() * 100), 2),
}

stats = pd.DataFrame([summary])
print(stats.to_string(index=False))

# Save summary + per-row lengths.
stats.to_csv('bpe_length_stats.csv', index=False)
bpe_lengths.to_frame('bpe_length').to_csv('bpe_lengths.csv', index=False)

# Download the files to your local machine. The browser downloads them to
# your default Downloads folder — move them into `outputs/results/` in the repo.
from google.colab import files
files.download('bpe_length_stats.csv')
files.download('bpe_lengths.csv')
print('\nDone. Move the 2 downloaded files into outputs/results/ in the repo.')